# Staged ingredient matching (10K RecipeNLG × `food_4macro`)

Iterate on a **two-stage hybrid** matcher:
1. **Base identity** — mostly lexical (+ small semantic) on parsed **name**
2. **Prep / version** — semantic + lexical prep, modifier rules, default/basic bonus

**One-time upstream steps** (cached under `scratch/recipe_matching_10k/`):
- Parse recipe lines and `food_4macro` descriptions with `ingredient-parser-nlp`
- Embed **name**, **preparation**, and **dequantified** text separately (`all-MiniLM-L6-v2`)
- Empty recipe preparation → shared **unprepared** embedding (one vector, embedded once)

Implementation: [`scripts/ingredient_match_staged.py`](../scripts/ingredient_match_staged.py), [`scripts/ingredient_query_cache.py`](../scripts/ingredient_query_cache.py)


In [11]:
import ast
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "scripts"))

from ingredient_match_staged import (
    StagedMatchConfig,
    StagedFoodIndex,
    match_ingredients_staged,
)
from ingredient_query_cache import (
    ensure_hf_token,
    load_or_build_food_artifacts,
    load_or_build_recipe_artifacts,
)
from progress_utils import force_std_tqdm

force_std_tqdm()  # text tqdm in Jupyter (no ipywidgets bars)
from load_food_4macro import load_food_4macro
from recipe_match_cache import (
    DEFAULT_CACHE_DIR,
    INGREDIENT_MATCHES_STAGED,
    RECIPE_SUMMARY_STAGED,
    load_or_run_ingredient_matches,
    load_or_run_summary,
)
import recipe_match_summary as recipe_match_metrics

summarize_recipe_matches = recipe_match_metrics.summarize_recipe_matches

WORK_DIR = DEFAULT_CACHE_DIR
WORK_DIR.mkdir(parents=True, exist_ok=True)

ensure_hf_token()  # HF_TOKEN from repo .env

# Tunable weights — edit and re-run §3–5 without rebuilding embeddings
MATCH_CONFIG = StagedMatchConfig()


## 1. Load recipes and explode ingredient lines

In [12]:
RECIPE_NLG_PATH = ROOT / "Data/recipes/RecipeNLG.csv"
RECIPE_NROWS = 10_000


def parse_ingredient_list(value) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).strip()
    if not text or text == "[]":
        return []
    if text.startswith("["):
        text = text[1:]
    if text.endswith("]"):
        text = text[:-1]
    text = text.strip()
    if not text:
        return []
    if '", "' not in text:
        item = text.strip('"')
        return [item] if item else []
    parts = text.split('", "')
    out: list[str] = []
    for i, part in enumerate(parts):
        part = part.strip()
        if i == 0:
            part = part.removeprefix('["').removeprefix('"')
        if i == len(parts) - 1:
            part = part.removesuffix('"]').removesuffix('"')
        if part:
            out.append(part)
    return out


recipes = pd.read_csv(RECIPE_NLG_PATH, nrows=RECIPE_NROWS)
recipes["ingredients_list"] = recipes["ingredients"].map(parse_ingredient_list)
recipes["recipe_id"] = recipes.index

recipe_ingredients = (
    recipes[["recipe_id", "ingredients_list"]]
    .explode("ingredients_list")
    .rename(columns={"ingredients_list": "ingredient"})
    .reset_index(drop=True)
)
recipe_ingredients["ingredient_idx"] = recipe_ingredients.groupby("recipe_id").cumcount()

print(f"recipes: {len(recipes):,}")
print(f"ingredient lines: {len(recipe_ingredients):,}")
recipe_ingredients.head(3)


recipes: 10,000
ingredient lines: 74,991


,recipe_id,ingredient,ingredient_idx
0,0,1 c. firmly packed brown sugar,0
1,0,1/2 c. evaporated milk,1
2,0,1/2 tsp. vanilla,2


## 2. One-shot parse + triple embeddings (cached)

Three vectors per recipe line and per `food_4macro` row: **name**, **preparation**, **dequantified** (parsed name/size/prep — quantity stripped).

Skipped when parquet + all six `.npy` files exist with matching row counts.

Long steps show **tqdm** progress bars (parsing, embedding, index build).


In [13]:
parsed_ingredients, name_embeddings, prep_embeddings, dequant_embeddings, recipe_meta = (
    load_or_build_recipe_artifacts(recipe_ingredients, WORK_DIR)
)

food_4macro_raw = load_food_4macro()
print(f"food_4macro rows: {len(food_4macro_raw):,}")

food_parsed, food_name_emb, food_prep_emb, food_dequant_emb, food_meta = load_or_build_food_artifacts(
    food_4macro_raw,
    WORK_DIR,
)

food_index = StagedFoodIndex.from_catalog(
    food_4macro_raw,
    name_embeddings=food_name_emb,
    prep_embeddings=food_prep_emb,
    dequant_embeddings=food_dequant_emb,
    config=MATCH_CONFIG,
    show_progress=True,
)

print("recipe embeddings:", recipe_meta)
if "prep_used_unprepared" in parsed_ingredients.columns:
    n_unprep = int(parsed_ingredients["prep_used_unprepared"].sum())
    print(f"recipe prep: {n_unprep:,} lines use unprepared proxy (empty preparation)")
print("food embeddings:", food_meta)
print(f"food index: {len(food_index.candidates):,} candidates")
parsed_ingredients[
  ["recipe_id", "ingredient", "name", "preparation", "prep_used_unprepared", "dequantified"]
].head(8)


Loaded cached recipe parse + embeddings (74,991 rows) → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k
food_4macro rows: 96,996
Loaded cached food_4macro parse + embeddings (96,996 rows) → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k


Building food index: 100%|██████████| 96996/96996 [00:02<00:00]

recipe embeddings: {'n_rows': 74991, 'name_shape': [74991, 384], 'prep_shape': [74991, 384], 'dequant_shape': [74991, 384], 'unprepared_prep_proxy_n': 58720, 'unprepared_prep_text': 'unprepared'}
recipe prep: 58,720 lines use unprepared proxy (empty preparation)
food embeddings: {'n_rows': 96996, 'name_shape': [96996, 384], 'prep_shape': [96996, 384], 'dequant_shape': [96996, 384]}
food index: 96,996 candidates


,recipe_id,ingredient,name,preparation,prep_used_unprepared,dequantified
0,0,1 c. firmly packed brown sugar,brown sugar,,True,brown sugar
1,0,1/2 c. evaporated milk,evaporated milk,,True,evaporated milk
2,0,1/2 tsp. vanilla,vanilla,,True,vanilla
3,0,1/2 c. broken nuts (pecans),broken nuts (pecans),,True,broken nuts (pecans)
4,0,2 Tbsp. butter or margarine,"butter, margarine",,True,"butter, margarine"
5,0,3 1/2 c. bite size shredded rice biscuits,rice biscuits,bite size shredded,False,"rice biscuits, bite size shredded"
6,1,"1 small jar chipped beef, cut up",beef,"chipped, cut up",False,"beef, chipped, cut up"
7,1,4 boned chicken breasts,chicken breasts,boned,False,"chicken breasts, boned"


## 3. Staged hybrid matching

**Stage 1 — base identity** (0.80 lexical / 0.20 semantic on name)  
**Stage 2 — prep/version** (semantic + lexical prep + modifier rules + default bonus)  
**Final** = 0.65×base + 0.25×prep + 0.10×default − unsupported-modifier penalties


In [14]:
MATCHES_PATH = WORK_DIR / INGREDIENT_MATCHES_STAGED
_n_ing = len(parsed_ingredients)


def _run_staged_match():
    return match_ingredients_staged(
        parsed_ingredients,
        name_embeddings,
        prep_embeddings,
        dequant_embeddings,  # required — dequantified ingredient text
        food_index,  # required — built in §2 from food_4macro embeddings
        show_progress=True,
    )


ingredient_matches = load_or_run_ingredient_matches(
    MATCHES_PATH,
    _run_staged_match,
    expected_rows=_n_ing,
    cache_key="ingredient_matches_staged",
    dir_path=WORK_DIR,
)

ingredient_matches[
    [
        "recipe_id",
        "name",
        "preparation",
        "matched_description",
        "match_score",
        "base_score",
        "prep_score",
        "modifier_penalty",
        "fallback_reason",
        "match_quality",
    ]
].head(12)


Cache miss — running matcher (will save → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/ingredient_matches_staged.csv)


Staged matching: 100%|██████████| 74991/74991 [01:31<00:00]


Saved → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/ingredient_matches_staged.csv (74,991 rows)


,recipe_id,name,preparation,matched_description,match_score,base_score,prep_score,modifier_penalty,fallback_reason,match_quality
0,0,brown sugar,,ORGANIC BROWN SUGAR,0.6624,0.7129,0.4362,0.0,neutral_default,medium
1,0,evaporated milk,,EVAPORATED MILK,0.7034,0.7760,0.4362,0.0,neutral_default,medium
2,0,vanilla,,"VANILLA BAR, VANILLA",0.6282,0.6602,0.4362,0.0,neutral_default,medium
3,0,broken nuts (pecans),,"PECAN NUT & RICE CRACKER SNACKS, PECAN",0.5255,0.5022,0.4362,0.0,neutral_default,low
4,0,"butter, margarine",,"Margarine-like, margarine-butter blend, soybea...",0.5560,0.6838,0.3662,0.0,base_only,medium
5,0,rice biscuits,bite size shredded,"SHREDDED WHEAT BIG BISCUIT CEREAL, SHREDDED WHEAT",0.3781,0.3925,0.1319,0.0,prep_fallback,unresolved
6,1,beef,"chipped, cut up","Beef, composite of trimmed retail cuts, separa...",0.7360,0.7919,0.4848,0.0,base_only,medium
7,1,chicken breasts,boned,CHICKEN BREAST,0.5882,0.7038,0.1628,0.0,prep_fallback,medium
8,1,cream of mushroom soup,,"Soup, cream of mushroom, canned, condensed",0.5114,0.6108,0.3775,0.0,base_only,low
9,1,sour cream,,"LIGHT SOUR CREAM, LIGHT",0.6520,0.6968,0.4362,0.0,neutral_default,medium


## 4. Aggregate quality

In [15]:
total = len(ingredient_matches)
resolved = ingredient_matches["matched_fdc_id"].notna().sum()
summary = pd.Series(
    {
        "match_rate_pct": round(100 * resolved / total, 2),
        "high": (ingredient_matches["match_quality"] == "high").sum(),
        "medium": (ingredient_matches["match_quality"] == "medium").sum(),
        "low": (ingredient_matches["match_quality"] == "low").sum(),
        "unresolved": (ingredient_matches["match_quality"] == "unresolved").sum(),
        "avg_match_score": round(ingredient_matches["match_score"].mean(), 4),
        "avg_base_score": round(ingredient_matches["base_score"].mean(), 4),
        "avg_prep_score": round(ingredient_matches["prep_score"].mean(), 4),
    }
)
display(summary.to_frame("value"))
display(ingredient_matches["match_quality"].value_counts().to_frame("count"))
display(ingredient_matches["fallback_reason"].value_counts(dropna=False).head(10).to_frame("count"))


,value
match_rate_pct,99.2900
high,2873.0000
medium,54633.0000
low,14907.0000
unresolved,2578.0000
avg_match_score,0.6029
avg_base_score,0.6636
avg_prep_score,0.3977


,count
match_quality,
medium,54633
low,14907
high,2873
unresolved,2578


,count
fallback_reason,
neutral_default,53438
prep_fallback,12159
base_only,4296
NaN,3457
weak_base,1606
unsupported_flavor_last_resort,35


## 5. Per-recipe summary

In [16]:
SUMMARY_PATH = WORK_DIR / RECIPE_SUMMARY_STAGED

recipe_match_summary = load_or_run_summary(
    SUMMARY_PATH,
    lambda: summarize_recipe_matches(ingredient_matches, recipes),
    expected_recipes=RECIPE_NROWS,
)

with_ing = recipe_match_summary.loc[recipe_match_summary["n_ingredients"] > 0]
display(
    with_ing[
        [
            "percent_high",
            "percent_medium",
            "percent_low",
            "percent_unmatched",
            "percent_med_high",
            "avg_match_score",
        ]
    ].mean().round(2)
)
recipe_match_summary.head(8)


Cache miss — building summary (will save → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/recipe_match_summary_staged.csv)
Saved → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/recipe_match_summary_staged.csv (10,000 recipes)


percent_high          3.63
percent_medium       72.63
percent_low          20.10
percent_unmatched     3.63
percent_med_high     76.26
avg_match_score       0.60
dtype: float64

,recipe_id,n_ingredients,percent_high,percent_medium,percent_low,percent_unmatched,percent_med_high,percent_high_100,avg_match_score,high,medium,low,unresolved
0,0,6,0.0,66.67,16.67,16.67,66.67,0.0,0.5756,0,4,1,1
1,1,4,0.0,75.00,25.00,0.00,75.00,0.0,0.6219,0,3,1,0
2,2,6,0.0,50.00,50.00,0.00,50.00,0.0,0.5505,0,3,3,0
3,3,5,0.0,60.00,40.00,0.00,60.00,0.0,0.6011,0,3,2,0
4,4,5,0.0,80.00,20.00,0.00,80.00,0.0,0.6362,0,4,1,0
5,5,10,0.0,70.00,30.00,0.00,70.00,0.0,0.5769,0,7,3,0
6,6,10,10.0,80.00,0.00,10.00,90.00,0.0,0.6019,1,8,0,1
7,7,6,0.0,50.00,50.00,0.00,50.00,0.0,0.5866,0,3,3,0


## 6. Staged hyperparameter grid search

**Two-phase grid** (evaluation is per-stage, not one blended score):

| Phase | Grid | Rank by | Metrics prefix |
|-------|------|---------|----------------|
| 1 Identity | `name_sem` × `dequant_sem` | `stage1_avg` (`base_score`) | `stage1_*`, `name_channel_*`, `dequant_channel_*` |
| 2 Prep | `prep_sem` (best identity fixed) | `stage2_avg` (`prep_score`) | `stage2_*` |

`final_*` columns are reported for reference only — **not** used to pick HP.

Outputs under `hp_sweep/`: leaderboards, per-config match CSVs, `hp_best_config.json`.


In [17]:
from ingredient_match_hp_sweep import (
    QUICK_SEMANTIC_GRID,
    STAGE1_RANK_KEY,
    STAGE2_RANK_KEY,
    hp_sweep_dir,
    run_staged_hp_grid_search,
)
from recipe_match_cache import (
    HP_BEST_CONFIG_JSON,
    HP_IDENTITY_LEADERBOARD,
    HP_PREP_LEADERBOARD,
)

HP_GRID = QUICK_SEMANTIC_GRID  # 3×3 identity + 3 prep = 12 runs; DEFAULT_SEMANTIC_GRID → 25 + 5 = 30

hp_result = run_staged_hp_grid_search(
    parsed_ingredients,
    name_embeddings,
    prep_embeddings,
    dequant_embeddings,
    food_index,
    work_dir=WORK_DIR,
    identity_grid=HP_GRID,
    prep_grid=HP_GRID,
    base_config=MATCH_CONFIG,
    force=False,
    show_progress=True,
)

hp_dir = hp_result["hp_dir"]
identity_lb = hp_result["identity_leaderboard"]
prep_lb = hp_result["prep_leaderboard"]
best_config = hp_result["best_config"]
print(f"HP dir: {hp_dir}")
print(f"Best config slug: {hp_result['best_payload']['config_slug']}")


Phase 1 — identity grid: 9 configs (rank by stage1_avg)


HP identity:   0%|          | 0/9 [00:00<?]

[1/9] | nS0.00_dS0.00_pS0.50 | stage1_avg=0.6407 | stage1_pct_gte_0_55=83.31 | name_ch_avg=0.6564 | dequant_ch_avg=0.6192


HP identity:  11%|█         | 1/9 [01:29<11:59]

[2/9] | nS0.00_dS0.50_pS0.50 | stage1_avg=0.6616 | stage1_pct_gte_0_55=87.38 | name_ch_avg=0.6546 | dequant_ch_avg=0.6936


...

[2/3] | nS1.00_dS1.00_pS0.50 | stage2_avg=0.4066 | stage2_pct_gte_0_55=4.55 | final_avg=0.6931


HP prep:  67%|██████▋   | 2/3 [02:49<01:24]

[3/3] | nS1.00_dS1.00_pS1.00 | stage2_avg=0.333 | stage2_pct_gte_0_55=1.92 | final_avg=0.6783


HP prep: 100%|██████████| 3/3 [05:13<00:00]


Prep leaderboard → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/hp_sweep/hp_prep_leaderboard.csv
Best config → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/hp_sweep/hp_best_config.json
HP dir: /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/hp_sweep
Best config slug: nS1.00_dS1.00_pS0.00


In [18]:
# Phase 1 leaderboard — identity / base stage only
identity_cols = [
    "config_slug",
    "base_name_semantic_weight",
    "base_dequant_semantic_weight",
    STAGE1_RANK_KEY,
    "stage1_pct_gte_0_55",
    "stage1_quality_high_pct",
    "name_channel_avg",
    "dequant_channel_avg",
    "matches_file",
]
display(identity_lb[identity_cols].head(12))


,config_slug,base_name_semantic_weight,base_dequant_semantic_weight,stage1_avg,stage1_pct_gte_0_55,stage1_quality_high_pct,name_channel_avg,dequant_channel_avg,matches_file
8,nS1.00_dS1.00_pS0.50,1.0,1.0,0.7955,93.18,61.95,0.8026,0.7982,ingredient_matches_identity_nS1.00_dS1.00_pS0....
7,nS1.00_dS0.50_pS0.50,1.0,0.5,0.7661,93.25,55.47,0.8016,0.7015,ingredient_matches_identity_nS1.00_dS0.50_pS0....
6,nS1.00_dS0.00_pS0.50,1.0,0.0,0.7379,92.16,45.11,0.8001,0.6103,ingredient_matches_identity_nS1.00_dS0.00_pS0....
5,nS0.50_dS1.00_pS0.50,0.5,1.0,0.7342,91.44,45.38,0.7161,0.7939,ingredient_matches_identity_nS0.50_dS1.00_pS0....
4,nS0.50_dS0.50_pS0.50,0.5,0.5,0.7065,90.13,37.33,0.7165,0.7001,ingredient_matches_identity_nS0.50_dS0.50_pS0....
2,nS0.00_dS1.00_pS0.50,0.0,1.0,0.6862,89.59,29.42,0.6528,0.7805,ingredient_matches_identity_nS0.00_dS1.00_pS0....
3,nS0.50_dS0.00_pS0.50,0.5,0.0,0.6808,89.84,26.74,0.7153,0.6165,ingredient_matches_identity_nS0.50_dS0.00_pS0....
1,nS0.00_dS0.50_pS0.50,0.0,0.5,0.6616,87.38,24.99,0.6546,0.6936,ingredient_matches_identity_nS0.00_dS0.50_pS0....
0,nS0.00_dS0.00_pS0.50,0.0,0.0,0.6407,83.31,14.22,0.6564,0.6192,ingredient_matches_identity_nS0.00_dS0.00_pS0....


In [19]:
# Phase 2 leaderboard — prep stage (identity weights fixed to phase-1 winner)
prep_cols = [
    "config_slug",
    "prep_semantic_weight",
    "prep_lexical_weight",
    STAGE2_RANK_KEY,
    "stage2_pct_gte_0_55",
    "stage2_quality_high_pct",
    "final_avg",
    "matches_file",
]
display(prep_lb[prep_cols].head(12))


,config_slug,prep_semantic_weight,prep_lexical_weight,stage2_avg,stage2_pct_gte_0_55,stage2_quality_high_pct,final_avg,matches_file
0,nS1.00_dS1.00_pS0.00,0.00,0.70,0.4890,9.89,6.74,0.7091,ingredient_matches_prep_nS1.00_dS1.00_pS0.00.csv
1,nS1.00_dS1.00_pS0.50,0.35,0.35,0.4066,4.55,3.28,0.6931,ingredient_matches_prep_nS1.00_dS1.00_pS0.50.csv
2,nS1.00_dS1.00_pS1.00,0.70,0.00,0.3330,1.92,1.60,0.6783,ingredient_matches_prep_nS1.00_dS1.00_pS1.00.csv


In [20]:
# Apply best HP to food index for §3-style matching (optional)
food_index.config = best_config
MATCH_CONFIG = best_config

best_matches_path = hp_dir / hp_result["best_payload"]["best_matches_file"]
print(f"Best match CSV for error analysis: {best_matches_path}")
print(f"Config JSON: {hp_dir / HP_BEST_CONFIG_JSON}")


Best match CSV for error analysis: /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/hp_sweep/ingredient_matches_prep_nS1.00_dS1.00_pS0.00.csv
Config JSON: /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k/hp_sweep/hp_best_config.json


## 7. Error analysis samples

In [21]:
# Fallback / low-confidence rows
fallback = ingredient_matches[
    ingredient_matches["fallback_reason"].notna()
    | (ingredient_matches["match_quality"].isin(["low", "unresolved"]))
].copy()

cols = [
    "recipe_id",
    "ingredient",
    "name",
    "preparation",
    "matched_description",
    "match_score",
    "base_score",
    "prep_score",
    "modifier_penalty",
    "fallback_reason",
    "match_quality",
]
display(fallback[cols].sample(20, random_state=0) if len(fallback) else fallback)

# High modifier penalty (likely over-specific food vs simple query)
penalized = ingredient_matches.nlargest(15, "modifier_penalty")
display(penalized[cols])


,recipe_id,ingredient,name,preparation,matched_description,match_score,base_score,prep_score,modifier_penalty,fallback_reason,match_quality
20021,2658,"2 c. graham crackers, crushed",graham crackers,crushed,GRAHAM CRACKERS,0.6246,0.7250,0.2532,0.0,prep_fallback,medium
52653,7030,1 Tbsp. vanilla,vanilla,,"VANILLA BAR, VANILLA",0.6282,0.6602,0.4362,0.0,neutral_default,medium
11537,1545,2 (16 oz.) cans lima beans,lima beans,,"Lima beans, immature seeds, frozen, baby, unpr...",0.7484,0.8259,0.4462,0.0,neutral_default,medium
20939,2782,1/2 cake yeast,cake yeast,,COFFEE CAKE,0.4676,0.4132,0.4362,0.0,neutral_default,low
51003,6808,vinegar,vinegar,,ORGANIC COCONUT VINEGAR,0.6105,0.6330,0.4362,0.0,neutral_default,medium
68195,9082,1/2 tsp. crushed basil leaves,basil leaves,crushed,HOLY BASIL TULSI LEAF ORGANIC SPICES,0.5207,0.5652,0.2532,0.0,prep_fallback,low
34389,4582,1/2 block margarine or butter,"margarine, butter",,"Margarine-like, margarine-butter blend, soybea...",0.5554,0.6829,0.3662,0.0,base_only,medium
53235,7109,1 c. mayonnaise,mayonnaise,,"Mayonnaise, reduced-calorie or diet, cholester...",0.6610,0.8453,0.3662,0.0,base_only,medium
17505,2333,1 egg,egg,,EGG ROLLS,0.6123,0.6357,0.4362,0.0,neutral_default,medium
10374,1381,1 c. rice,rice,,BLACK RICE,0.6362,0.6726,0.4362,0.0,neutral_default,medium


,recipe_id,ingredient,name,preparation,matched_description,match_score,base_score,prep_score,modifier_penalty,fallback_reason,match_quality
54114,7228,2 c. diced turkey,turkey,diced,"Turkey, diced, light and dark meat, seasoned",0.3022,0.8833,0.7922,0.49,NaN,unresolved
45104,6014,1 large bottle Sprite,Sprite,,"SPRITE, LEMON-LIME BEVERAGE MIX",0.2818,0.6465,0.3662,0.25,unsupported_flavor_last_resort,unresolved
223,33,"3 c. potatoes, shredded coarse",potatoes,shredded coarse,"Potatoes, yellow fleshed, hash brown, shredded...",0.5735,0.7642,0.8670,0.24,NaN,medium
418,63,1 1/2 c. uncooked rice,rice,uncooked,"Rice, white, glutinous, unenriched, uncooked",0.6648,0.8920,0.9000,0.24,NaN,medium
654,99,1 can Eagle Brand milk,Eagle Brand milk,,"EAGLE BRAND, BORDEN, SWEETENED CONDENSED MILK,...",0.2372,0.6010,0.2662,0.24,base_only,unresolved
757,111,1/2 c. cooked peas,peas,cooked,"Peas, green, cooked, boiled, drained, with salt",0.5186,0.8245,0.8106,0.24,NaN,low
970,141,1 can Eagle Brand milk,Eagle Brand milk,,"EAGLE BRAND, BORDEN, SWEETENED CONDENSED MILK,...",0.2372,0.6010,0.2662,0.24,base_only,unresolved
1010,146,1 (14 oz.) can Eagle Brand milk,Eagle Brand milk,,"EAGLE BRAND, BORDEN, SWEETENED CONDENSED MILK,...",0.2372,0.6010,0.2662,0.24,base_only,unresolved
1030,150,1 (14 oz.) Eagle Brand milk,Eagle Brand milk,,"EAGLE BRAND, BORDEN, SWEETENED CONDENSED MILK,...",0.2372,0.6010,0.2662,0.24,base_only,unresolved
1610,225,1/2 c. rice (uncooked),rice,(uncooked),"Rice, white, glutinous, unenriched, uncooked",0.6550,0.8913,0.8625,0.24,NaN,medium


In [ ]:
display(penalized[cols].head(5)[['ingredient', 'name', 'preparation', 'matched_description']])

Index(['recipe_id', 'ingredient_idx', 'ingredient', 'quantity',
       'quantity_max'],
      dtype='str')

In [ ]:
examples = ['4 chicken breasts, skin removed', '1 Tbsp vanilla', '2 (16 oz.) cans lima beans']